# 05 - Gradient Boosted Decision Trees (XGBoost)
Trains an XGBoost Regressor capable of learning non-linear threshold effects and peak demand behaviors.


In [ ]:
import xgboost as xgb
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from ml.preprocessing.feature_engineering import create_time_series_features, chronological_train_test_split

df = pd.read_csv('../data/processed/processed_energy_data.csv', parse_dates=['timestamp'], index_col='timestamp')
df_feat = create_time_series_features(df)

train_df, test_df = chronological_train_test_split(df_feat, test_size=0.2)
feats = ['hour_sin', 'hour_cos', 'dayofweek_sin', 'dayofweek_cos', 'is_weekend', 'lag_1', 'lag_2', 'lag_24', 'lag_168', 'rolling_mean_6h', 'rolling_mean_24h', 'rolling_std_24h']
feats = [f for f in feats if f in df_feat.columns]

X_train, y_train = train_df[feats], train_df['Global_active_power']
X_test, y_test = test_df[feats], test_df['Global_active_power']

xgb_model = xgb.XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.04, subsample=0.85, random_state=42)
xgb_model.fit(X_train, y_train)
preds = xgb_model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

print(f"XGBoost Regressor -> MAE: {mae:.4f} kW | RMSE: {rmse:.4f} kW | R2: {r2:.4f}")


### Feature Importance Extraction
Evaluating the gain and contribution of each lag feature.


In [ ]:
importances = pd.Series(xgb_model.feature_importances_, index=feats).sort_values(ascending=False)
print("Top Feature Importances:
", importances.head(8))


### Conclusion
XGBoost significantly outperforms the linear baseline, achieving superior R² score and capturing evening peak surges.
